# Magnetische Suszeptibilität 

## Was soll bestimmt werden und wie ?

Es soll die Volumensuszeptibilität $\chi_V$ von drei Proben bestimmt werden.
Es wurde ein Gewichtsunterschied gemessen.
Die notwendige Formel lautet:
$$
\chi_V = \frac{2 \mu_0 \Delta m g}{V_S \left(\frac{\partial B_z ^2}{\partial z} \right)}
$$
$V_S$ ist das Volumen der Probe, also 1 mm und 
dafür brauch man noch $\frac{\partial B_z ^2}{\partial z} $

$$
\frac{\partial B_z ^2}{\partial z} \approx \frac{B_z ^2(d+h_s) - B_z ^2(d) }{h_s}
$$
bzw.
$$
\frac{\partial B_z ^2}{\partial z} \approx \frac{B_r ^2}{4 \cdot h_s} \left(\left(\frac{D+d+h_s}{\sqrt{R^2 + (D + d + h_s)^2}} - \frac{d + h_s}{\sqrt{R^2 + (d + h_s)^2}} \right)^2 - \left(\frac{D+d}{\sqrt{R^2 + (D + d)^2}} - \frac{d}{\sqrt{R^2 + d^2}} \right)^2 \right)
$$

$h_s$ ist die Höhe der Probe.
$d$ ist der Abstand der Probe zum Magneten.

Wobei 
$$
B(z) = \frac{B_r}{2}\left(\frac{D+z}{\sqrt{R^2 + (D + z)^2}} - \frac{z}{\sqrt{R^2 + z^2}} \right)
$$
mit $B_r = 1,87 \pm 0,1 \,\mathrm{T}$
Der Radius von Neodynmagneten $r$ ist 30 mm.
Die Höhe D der Neodynmagenten ist 15 mm

Vorgehen:

    - Fkt. schreiben für B(z)
    - Fkt. für $\frac{\partial B_z ^2}{\partial z}$
    - Fkt. für $\chi_V$
    
    - Unsicherheiten für B(z) mit Fehlerfortpflanzung 
    - Unsicherheiten für $\frac{\partial B_z ^2}{\partial z}$ mit Fehlerfortpflanzung 
    - Unsicherheiten für $\chi_V$ mit Fehlerfortpflanzung 
    - Unsicherheiten der direkt gemessenen Größen

In [49]:
import numpy as np

# Konstanten 
pi = np.pi
mu_0 = 4*pi*1e-07
g = 9.81

# Werte "ohne" Unsicherheiten 

d = 0.001 # in m
D = 0.015 # in m
R = 0.03 # in m
B_r = 1.87 # in T
dm_halter = 0.0004 # in kg

In [50]:
def chi_V (dm, h, r):
    temp = ((2*mu_0 * dm* g)/(V_s(h,r) * p_B(h)))
    return temp

In [51]:
def p_B(h_s, d = d):
    temp = (B_z(d+h_s)**2 - B_z(d)**2)/h_s
    return temp

In [52]:
def B_z(z, D = D, B_r = B_r, R = R):
    temp = (B_r/2)*((D+z)/(np.sqrt(R**2 + (D+z)**2)) - (z)/(np.sqrt(R**2 + z**2)))
    return temp

In [53]:
# hatte alle die Form eines Zylinders
def V_s(h,r):
    return h*r**2*pi

# Unsicherheiten

- $\chi_V$ unsicherheitsbelastet durch $\Delta m$ und $\frac{\partial B_z ^2}{\partial z}$
- $\frac{\partial B_z ^2}{\partial z}$ unsicherheitsbelastet durch $B(z)$
- $B_z$ unsicherheitsbelastet durch $B_r$

In [54]:
u_B_r = 0.1 # in T
u_gerät = 0.00001 # Hersteller
u_ablese = 0.00001/(2*np.sqrt(3)) # Rechteck

u_dm_halter = np.sqrt((dm_halter*u_gerät)**2 + u_ablese**2)

### Unsicherheit für $B_z$ 
vllt. unnötig gewesen

$$
u_{B(z)} =  \frac{u_{B_r}}{2}\left(\frac{D+z}{\sqrt{R^2 + (D + z)^2}} - \frac{z}{\sqrt{R^2 + z^2}} \right)
$$

In [55]:
def calc_u_B_z(z, D = D, B_r = B_r, R = R, u_B_r = u_B_r):
    temp = (u_B_r/2)*((D+z)/(np.sqrt(R**2 + (D+z)**2)) - (z)/(np.sqrt(R**2 + z**2)))
    return temp

### Unsicherheit für $\frac{\partial B_z ^2}{\partial z}$
bei $z = d + h_s$
$$
u_{\frac{\partial B_z ^2}{\partial z}} \approx \frac{B_r}{2 \cdot h_s} \left(\left(\frac{D+d+h_s}{\sqrt{R^2 + (D + d + h_s)^2}} - \frac{d + h_s}{\sqrt{R^2 + (d + h_s)^2}} \right)^2 - \left(\frac{D+d}{\sqrt{R^2 + (D + d)^2}} - \frac{d}{\sqrt{R^2 + d^2}} \right)^2 \right)\cdot u_{B_r}
$$
$$
= \frac{2}{B_r h_s} (B_z (d+h_s)^2 - B_z(d)^2)\cdot u_{B_r}
$$

In [56]:
def calc_u_p_B(h_s, d = d, B_r = B_r, u_B_r = u_B_r):
    temp = ((2*u_B_r)/(B_r*h_s))*(B_z(d + h_s)**2 - B_z(d)**2)
    return temp

### Unsicherheit für $\chi_V$
unsicherheitsbelastete Größen $\Delta m$ und $\frac{\partial B_z ^2}{\partial z}$
$$
u_{\chi_V} = \sqrt{\left(\frac{2 \mu_0 g}{V_S \left(\frac{\partial B_z ^2}{\partial z} \right)} u_{\Delta m} \right)^2 
+ \left( - \frac{2 \mu_0 \Delta m g}{V_S \left(\frac{\partial B_z ^2}{\partial z} \right)^2} \cdot u_{\frac{\partial B_z ^2}{\partial z}}\right)^2}
$$

In [57]:
def calc_u_chi(dm, u_dm, h, r, g = g, mu_0 = mu_0):
    temp1 = (2 * mu_0 * u_dm)/(V_s(h,r)*p_B(h))
    temp2 = (2 * mu_0 * dm * g)/(V_s(h, r)*p_B(h)**2)
    return (np.sqrt(temp1**2 + temp2**2))

## Aluminium 

In [75]:
dm_alu_mess = 0.00034 # in kg
u_dm_alu_mess = np.sqrt((dm_alu_mess*u_gerät)**2 + u_ablese**2)
dm_alu = dm_alu_mess - dm_halter
u_dm_alu = np.sqrt(u_dm_alu_mess**2 + u_dm_halter**2)
r_alu = 0.02 # in m
h_alu = 0.005 # in m




chi_alu = chi_V(dm_alu, h_alu, r_alu)
u_chi_alu = calc_u_chi(dm_alu, u_dm_alu, h_alu, r_alu)

print(chi_alu)
print(u_chi_alu)
print()
print(u_dm_alu_mess)
print(u_dm_halter)
print(dm_alu)
print(u_dm_alu)

2.7583426373802252e-05
3.237247620646766e-06

2.8867533481981687e-06
2.886754117228091e-06
-5.9999999999999995e-05
4.082486280034101e-06


## pyrolytisches Graphit

In [72]:
dm_gra_mess = 0.00181 # in kg
u_dm_gra_mess = np.sqrt((dm_gra_mess*u_gerät)**2 + u_ablese**2)
dm_gra = dm_mess_gra - dm_halter
u_dm_gra = np.sqrt(u_dm_gra_mess**2 + u_dm_halter**2)
r_gra = 0.02 # in m
h_gra = 0.005 # in m

chi_gra = chi_V(dm_gra, h_gra, r_gra)
u_chi_gra = calc_u_chi(dm_gra, u_dm_gra, h_gra, r_gra)

print(chi_gra)
print(u_chi_gra)
print()
print(u_dm_gra_mess)
print(u_dm_halter)
print(dm_gra)
print(u_dm_gra)

-0.0006482105197843529
7.59425921078158e-05

2.886808089106953e-06
2.886754117228091e-06
0.00141
4.082524987880254e-06


## Glas

In [73]:
dm_mess_glas = 0.00042 # in kg
u_dm_mess_glas = np.sqrt((dm_mess_glas*u_gerät)**2 + u_ablese**2)

dm_halter_glas = 0.00039 # in kg
u_dm_halter_glas = np.sqrt((dm_halter_glas*u_gerät)**2 + u_ablese**2)

dm_glas = dm_mess_glas - dm_halter_glas
u_dm_glas = np.sqrt(u_dm_mess_glas**2 + u_dm_halter_glas**2)
r_glas = 0.02 # in m
h_glas = 0.008 # in m

chi_glas = chi_V(dm_glas, h_glas, r_glas)
u_chi_glas = calc_u_chi(dm_glas, u_dm_glas, h_glas, r_glas)

print(chi_glas)
print(u_chi_glas)
print()
print(u_dm_mess_glas)
print(u_dm_halter_glas)
print(dm_glas)
print(u_dm_glas)

-8.698181240793097e-06
1.0353708565951222e-06

2.886754401284137e-06
2.8867539803962056e-06
3.0000000000000024e-05
4.0824869279235505e-06
